In [1]:
import numpy as np

from utils.dgp import DataGenerator, heiss_x_sampler, heiss_beta_support_probs
from utils.estimators import FKRBEstimator
from utils.visualization import plot_cdf_3D

In [2]:
J = 4
K = 2
N_SET = (1000, 10000)
R_SET_DISCRETE = (25, 81, 289)
S_SET_DISCRETE = (17, 49, 161)

OUTER_REPETITIONS = 200
INNER_REPETITIONS = 50

SEED = 42

rng = np.random.default_rng(SEED)


In [5]:
def get_mean(support, probs):
    return support @ probs

for N in N_SET:
    for R in R_SET_DISCRETE:
        full_grid, support, probs = heiss_beta_support_probs(R)
        generator = DataGenerator(N=N, J=J, K=K, rng=rng)
        y, x = generator.generate(x_sampler=heiss_x_sampler, beta_support=support, beta_probs=probs)

        estimator = FKRBEstimator(beta_support=full_grid)
        theta = estimator.estimate(y=y, x=x)

        ci = estimator.get_confidence_interval(alpha=0.5)

        support_1 = support[:,0]
        support_2 = support[:,1]

        estimated_mean_1, ci_1 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_1,probs), ci = True)
        estimated_mean_2, ci_2 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_2,probs), ci = True)
        
        coverage_rates = np.zeros(INNER_REPETITIONS)
        coverage_rates_1 = np.zeros(INNER_REPETITIONS)
        coverage_rates_2 = np.zeros(INNER_REPETITIONS)
        for i in range(INNER_REPETITIONS):
            generator.reset()
            y, x = generator.generate(x_sampler=heiss_x_sampler, beta_support=support, beta_probs=probs)

            estimator = FKRBEstimator(beta_support=full_grid)
            theta = estimator.estimate(y=y, x=x, estimate_both=False)
            estimated_mean_1 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_1,probs), ci = False)
            estimated_mean_2 = estimator.plug_in_estimate_functional(lambda probs: get_mean(support_2,probs), ci = False)

            in_interval = (theta >= ci[:, 0]) & (theta <= ci[:, 1])
            in_interval_1 = (estimated_mean_1 >= ci_1[0]) & (estimated_mean_1 <= ci_1[1])
            in_interval_2 = (estimated_mean_2 >= ci_2[0]) & (estimated_mean_2 <= ci_2[1])

            coverage_rates[i] = np.mean(in_interval)
            coverage_rates_1[i] = in_interval_1
            coverage_rates_2[i] = in_interval_2
        
        print("regular")
        print(np.mean(coverage_rates))
        print("first variable")
        print(np.mean(coverage_rates_1))
        print("second variable")
        print(np.mean(coverage_rates_2))
            

regular
0.8048000000000001
first variable
1.0
second variable
1.0
regular
0.9997530864197531
first variable
1.0
second variable
1.0
regular
0.9999307958477509
first variable
1.0
second variable
1.0
regular
0.6464
first variable
1.0
second variable
1.0
regular
0.9995061728395062
first variable
1.0
second variable
1.0


KeyboardInterrupt: 